# Vision Transformers

#Steps:
Patch extraction

1.   Patch embedding
2. CLS token
3. Positional embeddings
4. Single-head attention first
5. Multi-head attention
6. MLP block
7. One Transformer encoder block
8. Stack 12 encoder blocks
9. Classification head
10. Training loop
11. Compare with 97% CNN


##1. patch extarction

*   load the image
*   make the patches for every image
the size of the image patches like that
[channels,patch rows,patch rows, patch columns, patch height , patch width]
*   flatten the patches so they are like the transformers tokens







#2. CLS token
CLS token as a learned summary representation of the whole image
it has the same size as the other tokens the features that will be in it will the most important ones that we get from the other tokens so . it contains information about all the other tokens

##3. positional embedding:
in here the positional embeddings are the position informatin that the transformer wants to know the position of each token the original transformer they provided as give not learnable but in vit they thought that lets make it a learnable as maybe the model think this representation are not good for the model

## 4. Single-head attention
the idea behind it is that for all the patches i have to make their q ,k and value vector but at this point we need to just make it as learnable parameter

define a class for vision transformer

In [1]:
from google.colab import files
files.upload()


Saving leaves (3).zip to leaves (3).zip
unzip:  cannot find or open leaves (1).zip, leaves (1).zip.zip or leaves (1).zip.ZIP.


In [4]:
!unzip "leaves (3).zip" -d dataset

Streaming output truncated to the last 5000 lines.
  inflating: dataset/images/Apple_healthy/image (720).JPG  
  inflating: dataset/images/Apple_healthy/image (721).JPG  
  inflating: dataset/images/Apple_healthy/image (722).JPG  
  inflating: dataset/images/Apple_healthy/image (723).JPG  
  inflating: dataset/images/Apple_healthy/image (724).JPG  
  inflating: dataset/images/Apple_healthy/image (725).JPG  
  inflating: dataset/images/Apple_healthy/image (726).JPG  
  inflating: dataset/images/Apple_healthy/image (727).JPG  
  inflating: dataset/images/Apple_healthy/image (728).JPG  
  inflating: dataset/images/Apple_healthy/image (729).JPG  
  inflating: dataset/images/Apple_healthy/image (73).JPG  
  inflating: dataset/images/Apple_healthy/image (730).JPG  
  inflating: dataset/images/Apple_healthy/image (731).JPG  
  inflating: dataset/images/Apple_healthy/image (732).JPG  
  inflating: dataset/images/Apple_healthy/image (733).JPG  
  inflating: dataset/images/Apple_healthy/image (7

In [99]:
from torch import nn
import math
class VisionTransformer(nn.Module):
  def __init__(self,patch_size = 16,channels = 3,embed_dim = 512, image_size=224,head_nums= 8,B=32):
    super().__init__()
    self.patch_size = patch_size
    self.patch_dim = channels * patch_size * patch_size
    self.patch_nums= (image_size//patch_size)**2
    self.head_nums = head_nums
    self.embed_dim = embed_dim
    self.B = B

    self.patch_embeddings= nn.Linear(self.patch_dim , self.embed_dim )
    self.cls = nn.Parameter(torch.randn(1,1,self.embed_dim))
    self.POS = nn.Parameter(torch.randn(1,self.patch_nums+1,self.embed_dim))
    self.WQ = nn.Linear(self.embed_dim,self.embed_dim)
    self.WK = nn.Linear(self.embed_dim,self.embed_dim)
    self.WV = nn.Linear(self.embed_dim,self.embed_dim)
    self.WO = nn.Linear(self.embed_dim,self.embed_dim)
    self.layer_norm1 = nn.LayerNorm(self.embed_dim)
    self.layer_norm2 = nn.LayerNorm(self.embed_dim)
    self.mlp = nn.Sequential(
        nn.Linear(self.embed_dim,self.embed_dim*4),
        nn.GELU(),
        nn.Linear(self.embed_dim*4,self.embed_dim)
    )
    self.classifier = nn.Linear(self.embed_dim,8)

  def make_patches(self,images):
    """ take the image and patch it pased on the patch size and return the flatten patches """
    print(images.shape)
    B, C, H, W = images.shape
    patches = images.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
    patches=patches.permute(0,2,3,1,4,5)
    patches = patches.reshape(B, self.patch_nums,-1)
    return patches

  def encoder(self,patches):


    # layer norm
    patches = self.layer_norm1(patches)
    # multi head attention
    Q = self.WQ(patches)
    K = self.WK(patches)
    V = self.WV(patches)
    # Multi-head attention splits the Q, K, and V feature dimensions into multiple heads
    # so that the model can learn multiple attention relationships between the patches in parallel.
    Q =Q.reshape(self.B,self.patch_nums+1,self.head_nums,-1)
    print("Q before",Q.shape)
    Q =Q.permute(0,2,1,3)
    K =K.reshape(self.B,self.patch_nums+1,self.head_nums,-1)
    K =K.permute(0,2,1,3)
    V =V.reshape(self.B,self.patch_nums+1,self.head_nums,-1)
    V =V.permute(0,2,1,3)
    scores = torch.matmul(Q,K.transpose(-2,-1))
    # the next step for scaling the score so we are avoinding the gradient problems
    attention = scores/math.sqrt(self.embed_dim//self.head_nums)
    attention = torch.softmax(attention,dim=-1)
    output = torch.matmul(attention,V)
    output = output.permute(0,2,1,3)


    # collecting the features again together
    output = output.reshape(self.B,self.patch_nums+1,-1)
    output = self.WO(output)

    # residual connection
    patches = patches + output
    print("all_patches",patches.shape)
    # layer norm
    patches = self.layer_norm2(patches)
    # mlp
    patches = self.mlp(patches)
    # residual connection
    patches = patches + output
    print ("all_patches", patches.shape)
    return patches

  def foreward(self,x):
    patches = self.make_patches(x)
    patches_embbedding = self.patch_embeddings(patches)
    print(patches_embbedding.shape)
    cls = self.cls.expand(self.B,-1,-1)
    all_patches = torch.cat((cls,patches_embbedding),dim=1)
    pos=self.POS.expand(self.B,-1,-1)
    all_patches = all_patches + pos

    # six encoder blocks
    all_patches = self.encoder(all_patches)
    all_patches = self.encoder(all_patches)
    all_patches = self.encoder(all_patches)
    all_patches = self.encoder(all_patches)
    all_patches = self.encoder(all_patches)
    all_patches = self.encoder(all_patches)
    print ("all patches after the encoder shape ",all_patches.shape)
    cls_token = all_patches[:,0]
    cls_token = self.classifier(cls_token)
    print("cls token shape",cls_token.shape)
    return cls_token













In [100]:
from PIL import Image
from torchvision.datasets import ImageFolder
from torchvision import transforms, torch
from torch.utils.data import random_split
from torch.utils.data import DataLoader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
dataset = ImageFolder(root="dataset/images", transform=transform)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])



In [101]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [102]:
images,labels = next(iter(train_loader))
print(images.shape)

torch.Size([32, 3, 224, 224])


In [103]:
model = VisionTransformer(patch_size=16)
model.foreward(images)

torch.Size([32, 3, 224, 224])
torch.Size([32, 196, 512])
Q before torch.Size([32, 197, 8, 64])
all_patches torch.Size([32, 197, 512])
all_patches torch.Size([32, 197, 512])
Q before torch.Size([32, 197, 8, 64])
all_patches torch.Size([32, 197, 512])
all_patches torch.Size([32, 197, 512])
Q before torch.Size([32, 197, 8, 64])
all_patches torch.Size([32, 197, 512])
all_patches torch.Size([32, 197, 512])
Q before torch.Size([32, 197, 8, 64])
all_patches torch.Size([32, 197, 512])
all_patches torch.Size([32, 197, 512])
Q before torch.Size([32, 197, 8, 64])
all_patches torch.Size([32, 197, 512])
all_patches torch.Size([32, 197, 512])
Q before torch.Size([32, 197, 8, 64])
all_patches torch.Size([32, 197, 512])
all_patches torch.Size([32, 197, 512])
all patches after the encoder shape  torch.Size([32, 197, 512])
cls token shape torch.Size([32, 8])


tensor([[ 0.1030, -0.2110, -0.3633,  0.2244, -0.1381, -0.2404, -0.0262,  0.3380],
        [ 0.1063, -0.2046, -0.3684,  0.2003, -0.1465, -0.2480, -0.0305,  0.3314],
        [ 0.1231, -0.2021, -0.3510,  0.1899, -0.1369, -0.2307, -0.0423,  0.3240],
        [ 0.1134, -0.2034, -0.3519,  0.2206, -0.1282, -0.2364, -0.0342,  0.3319],
        [ 0.1049, -0.2176, -0.3706,  0.2142, -0.1464, -0.2413, -0.0241,  0.3375],
        [ 0.1259, -0.1923, -0.3489,  0.1715, -0.1411, -0.2248, -0.0471,  0.3249],
        [ 0.1218, -0.1951, -0.3468,  0.1934, -0.1315, -0.2257, -0.0465,  0.3276],
        [ 0.1276, -0.1952, -0.3515,  0.1600, -0.1434, -0.2274, -0.0482,  0.3206],
        [ 0.0993, -0.2231, -0.3691,  0.2214, -0.1442, -0.2426, -0.0223,  0.3431],
        [ 0.1094, -0.2245, -0.3698,  0.2230, -0.1382, -0.2479, -0.0260,  0.3379],
        [ 0.1185, -0.2123, -0.3551,  0.2082, -0.1380, -0.2350, -0.0370,  0.3356],
        [ 0.1184, -0.2018, -0.3509,  0.1944, -0.1361, -0.2342, -0.0421,  0.3324],
        [ 0.1185

In [ ]:
for batch in train_loader:
  print(batch[0].shape)